# 07 - LoRA 微调对比: 手写 LoRALinear vs PEFT

对比 from-scratch LoRA (手写 LoRALinear 类) 与 PEFT 库的实现差异。

| 维度 | from-scratch | HuggingFace (PEFT) |
|------|-------------|--------------------|
| 实现 | `LoRALinear(nn.Module)` 手写 | `get_peft_model()` 自动注入 |
| 应用 | `apply_lora()` 逐层替换 | `get_peft_model(model, config)` |
| 合并 | `merge_lora()` 手动 W += BA | `model.merge_and_unload()` |
| 保存 | `lora_state_dict()` 提取参数 | `model.save_pretrained()` |
| 加载 | `load_lora_state_dict()` | `PeftModel.from_pretrained()` |
| QLoRA | 不支持 | BitsAndBytesConfig + LoRA |

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import tempfile
from pathlib import Path

from model import ClearMindConfig, ClearMindForCausalLM
from training.lora import create_lora_config, apply_peft_lora, merge_lora_weights

## 1. LoRA 应用对比

In [ ]:
config = ClearMindConfig.tiny()
model = ClearMindForCausalLM(config)
total = sum(p.numel() for p in model.parameters())
print(f'原始模型参数: {total:,}\n')

# === PEFT 方式 ===
lora_config = create_lora_config({'r': 8, 'lora_alpha': 16, 'lora_dropout': 0.0})
peft_model = apply_peft_lora(model, lora_config)

# 对比: from-scratch
# apply_lora(model, rank=8, alpha=16, target_modules=['w_q', 'w_k', 'w_v', 'w_o'])

In [ ]:
peft_model.print_trainable_parameters()

## 2. 合并与保存/加载

In [ ]:
input_ids = torch.randint(0, config.vocab_size, (1, 16))
peft_model.eval()

with torch.no_grad():
    before = peft_model(input_ids).logits

merged = merge_lora_weights(peft_model)
with torch.no_grad():
    after = merged(input_ids).logits

print(f'合并前后输出一致: {torch.allclose(before, after, atol=1e-5)}')
print(f'合并后类型: {type(merged).__name__}')

# 对比: from-scratch
# merge_lora(model)  # W.data += (B @ A) * scaling

## 3. LoRA + SFT 训练

In [ ]:
print('=== HuggingFace ===')
print('python scripts/train.py --stage sft --config configs/tiny.yaml --use_lora')
print()
print('=== from-scratch ===')
print('python scripts/train.py --stage sft --config configs/tiny.yaml --lora')

## 4. QLoRA — 4-bit 量化 + LoRA

QLoRA 是 LoRA 的扩展，将基础模型量化为 4-bit 后再应用 LoRA，显存占用约为全精度的 1/4。

**from-scratch 版没有 QLoRA 支持**，这是 HuggingFace 生态的独有优势。

| 对比 | LoRA | QLoRA |
|------|------|-------|
| 基础模型精度 | FP32/FP16 | 4-bit NF4 |
| LoRA 参数精度 | FP32/FP16 | FP16/BF16 |
| 显存占用 (7B) | ~14GB | ~4GB |
| 训练效果 | 基线 | 接近 LoRA |
| 硬件要求 | GPU | CUDA GPU + bitsandbytes |

In [ ]:
# === QLoRA 使用示例 (需要 CUDA GPU) ===
print('''
from training.lora import create_qlora_config, load_model_qlora

# 1. 创建 QLoRA 配置
bnb_config, lora_config = create_qlora_config({
    "r": 8, "lora_alpha": 16, "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
})

# 2. 加载量化模型 + 应用 LoRA
model = load_model_qlora("outputs/sft", bnb_config, lora_config)
# → 模型权重量化为 4-bit NF4
# → LoRA 参数为 FP16/BF16
# → 显存占用约为全精度的 1/4

# 3. 正常训练 (只更新 LoRA 参数)
trainer = Trainer(model=model, args=training_args, ...)
trainer.train()

# 注意: QLoRA 模型不能直接 merge_and_unload (量化权重)
# 保存 adapter 后，可在全精度模型上加载 adapter 再合并
''')

## 总结

| 功能 | from-scratch | PEFT |
|------|-------------|------|
| 实现 | `LoRALinear` ~100 行 | `get_peft_model()` 一行 |
| 合并 | `W += B@A * scaling` | `merge_and_unload()` |
| 保存 | `lora_state_dict()` | `save_pretrained()` 含 config |
| QLoRA | 不支持 | `BitsAndBytesConfig` + LoRA |
| 代码量 | ~210 行 | ~90 行 |